# MeshVTON: Full 3D Virtual Try-On Training

This notebook runs the full pipeline end-to-end:
1. GPU and library setup
2. Mount Google Drive for data
3. Extract SMPL-X body parameters
4. Render 3D garments
5. Train the model

**Requirements**: GPU runtime (T4 / A100 / H100)

---
## 1. GPU Check

In [ ]:
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('No GPU detected. Runtime -> Change runtime type -> GPU')

---
## 2️⃣ Projeyi Clone Et

In [ ]:
import os

# === EDIT THIS: your own GitHub repo URL ===
REPO_URL = 'https://github.com/SerhanTelatar/MeshVTON.git'
PROJECT_DIR = '/content/MeshVTON'

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
    print('Repo cloned')
else:
    !cd {PROJECT_DIR} && git pull
    print('Repo updated')

os.chdir(PROJECT_DIR)
print(f'Working directory: {os.getcwd()}')

---
## 3. Install Libraries

In [ ]:
# Core dependencies
!pip install -q omegaconf accelerate transformers diffusers
!pip install -q opencv-python-headless pillow scipy
!pip install -q lpips einops timm

# 3D pipeline dependencies
!pip install -q smplx trimesh pyrender

# PyTorch3D (may require CUDA compilation)
import sys
import torch
pyt_version = torch.__version__.split('+')[0]
cuda_version = torch.version.cuda.replace('.', '')

try:
    import pytorch3d
    print(f'PyTorch3D already installed: {pytorch3d.__version__}')
except ImportError:
    print('Installing PyTorch3D (this may take a few minutes)...')
    !pip install -q "git+https://github.com/facebookresearch/pytorch3d.git"
    print('PyTorch3D installed')

---
## 4. Mount Google Drive and Extract Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# === EDIT THIS: your Drive folder path ===
DRIVE_DATA = '/content/drive/MyDrive/MeshVTON'

import os
print('Drive contents:')
for f in os.listdir(DRIVE_DATA):
    size = os.path.getsize(os.path.join(DRIVE_DATA, f)) / 1e6
    print(f'  {f} ({size:.1f} MB)')

In [ ]:
import zipfile
import shutil
from pathlib import Path

PROJECT = Path('/content/MeshVTON')
DRIVE = Path(DRIVE_DATA)

def extract_zip(zip_name, target_dir):
    """Extract a zip file from Drive."""
    zip_path = DRIVE / zip_name
    if not zip_path.exists():
        print(f'  {zip_name} not found, skipping')
        return
    target = Path(target_dir)
    target.mkdir(parents=True, exist_ok=True)
    print(f'  Extracting {zip_name} -> {target_dir}...')
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(target)
    print(f'  {zip_name} done')

# Create the directory structure
for d in ['data/raw/images', 'data/processed/poses', 'data/processed/segments',
          'data/processed/agnostic',
          'data/processed/smplx_params', 'data/processed/renders_3d',
          'data/processed/normal_maps', 'data/processed/depth_maps',
          'data/garments_3d', 'checkpoints/pretrained']:
    (PROJECT / d).mkdir(parents=True, exist_ok=True)

print('Extracting zip files...')
print()

# Person images
extract_zip('images.zip', PROJECT / 'data/raw/images')

# Preprocessing outputs
extract_zip('poses.zip', PROJECT / 'data/processed')
extract_zip('segments.zip', PROJECT / 'data/processed')
extract_zip('agnostic.zip', PROJECT / 'data/processed')

# 3D garment meshes
extract_zip('garments_3d.zip', PROJECT / 'data/garments_3d')

# SMPL-X + VPoser
extract_zip('pretrained.zip', PROJECT / 'checkpoints')

# Copy CSV files
for csv_name in ['train_pairs.csv', 'val_pairs.csv', 'test_pairs.csv']:
    src = DRIVE / csv_name
    if src.exists():
        shutil.copy2(src, PROJECT / 'data/raw' / csv_name)

print()
print('All data loaded')

In [ ]:
# Sanity check
from pathlib import Path
P = Path('/content/MeshVTON')

checks = {
    'Person images': len(list((P/'data/raw/images').glob('*.jpg'))),
    'Pose': len(list((P/'data/processed/poses').glob('*'))),
    'Segmentation': len(list((P/'data/processed/segments').glob('*'))),
    'Agnostic': len(list((P/'data/processed/agnostic').glob('*'))),
    '3D garments (upper)': len(list((P/'data/garments_3d/upper_body').glob('*'))) if (P/'data/garments_3d/upper_body').exists() else 0,
    '3D garments (lower)': len(list((P/'data/garments_3d/lower_body').glob('*'))) if (P/'data/garments_3d/lower_body').exists() else 0,
    '3D garments (dress)': len(list((P/'data/garments_3d/dresses').glob('*'))) if (P/'data/garments_3d/dresses').exists() else 0,
    '3D garments (outer)': len(list((P/'data/garments_3d/outerwear').glob('*'))) if (P/'data/garments_3d/outerwear').exists() else 0,
    'SMPL-X model': (P/'checkpoints/pretrained/smplx/SMPLX_NEUTRAL.npz').exists(),
    'train_pairs.csv': (P/'data/raw/train_pairs.csv').exists(),
}

print('Data check:')
print('=' * 40)
for name, val in checks.items():
    status = 'OK' if val else 'MISSING'
    print(f'{status:8s} {name}: {val}')

---
## 5. Extract SMPL-X Body Parameters

Estimate the 3D body parameters (shape, pose) from each person image.

In [ ]:
import sys
sys.path.insert(0, '/content/MeshVTON')

from src.data.preprocessing.extract_smplx import extract_smplx

extract_smplx(
    image_dir='data/raw/images',
    output_dir='data/processed/smplx_params',
    model_dir='checkpoints/pretrained/smplx',
    device='cuda',
    save_mesh=True,
    mesh_dir='data/processed/smplx_meshes',
)
print('SMPL-X parameter extraction complete')

---
## 6️⃣ 3D Giysi Render

CLOTH3D mesh'lerini SMPL-X beden modellerine giydirip 2D'ye render eder.

In [ ]:
from src.data.preprocessing.render_garment import render_garments

render_garments(
    garments_dir='data/garments_3d',
    smplx_params_dir='data/processed/smplx_params',
    output_dir='data/processed/renders_3d',
    normal_maps_dir='data/processed/normal_maps',
    depth_maps_dir='data/processed/depth_maps',
    resolution=512,
    device='cuda',
)
print('3D garment rendering complete')

---
## 7. Start Training

In [ ]:
# Training config (sized for 16GB VRAM)
TRAIN_CONFIG = {
    'batch_size': 2,
    'gradient_accumulation_steps': 8,  # effective batch = 16
    'learning_rate': 1e-5,
    'num_epochs': 50,
    'mixed_precision': True,
    'gradient_checkpointing': True,
    'save_every': 5,   # checkpoint every 5 epochs
    'log_every': 100,  # log every 100 steps
}

print('Training configuration:')
for k, v in TRAIN_CONFIG.items():
    print(f'  {k}: {v}')

In [ ]:
# Start training
!python scripts/train.py \
    --config configs/train.yaml \
    --batch_size 2 \
    --gradient_accumulation 8 \
    --lr 1e-5 \
    --epochs 50 \
    --mixed_precision \
    --gradient_checkpointing \
    --save_dir checkpoints/runs

---
## 8. Save Checkpoints to Drive

In [ ]:
import shutil
from pathlib import Path

# Copy checkpoints to Drive so they survive Colab disconnects
src_ckpt = Path('checkpoints/runs')
dst_ckpt = Path(DRIVE_DATA) / 'checkpoints'
dst_ckpt.mkdir(parents=True, exist_ok=True)

if src_ckpt.exists():
    for f in src_ckpt.glob('*.pt'):
        shutil.copy2(f, dst_ckpt / f.name)
        print(f'Saved {f.name} -> Drive')

# Also save preprocessing outputs so we do not have to redo them
for folder in ['smplx_params', 'renders_3d', 'normal_maps', 'depth_maps']:
    src = Path(f'data/processed/{folder}')
    if src.exists() and any(src.iterdir()):
        dst = Path(DRIVE_DATA) / f'{folder}.zip'
        if not dst.exists():
            shutil.make_archive(str(dst).replace('.zip',''), 'zip', src)
            print(f'Saved {folder} -> Drive')

print('\nAll results saved to Google Drive')

---
## 9. Quick Test (Optional)

Run a try-on test on a single image with the trained model.

In [ ]:
from src.inference.image_tryon import ImageTryOn
from PIL import Image
import matplotlib.pyplot as plt

# Load the latest checkpoint
tryon = ImageTryOn(
    checkpoint_path='checkpoints/runs/latest.pt',
    device='cuda'
)

# Test inputs
person_img = 'data/raw/images/00001_00.jpg'
garment_mesh = 'data/garments_3d/upper_body/00047_Top/mesh.obj'

result = tryon.run(person_img, garment_mesh)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(Image.open(person_img));    axes[0].set_title('Person')
axes[1].imshow(result['render_3d']);       axes[1].set_title('3D Render')
axes[2].imshow(result['output']);          axes[2].set_title('Result')
for ax in axes: ax.axis('off')
plt.tight_layout()
plt.savefig('test_result.png', dpi=150)
plt.show()